In [0]:
import os
import numpy as np
from pathlib import Path
import joblib
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score
)
import argparse

In [0]:
ambiente = 'project'

In [0]:
PORCENTAJE_SAMPLE_DATA = 0.7
EVERY_N_YEARS = 2
RANDOM_SEED = 0
TEST_SIZE = 0.25

SCALER_FEATURES = [
    'wind_speed_ms', 'wind_cos_direction', 'wind_sin_direction', 'wave_height_m', 
    'wave_cos_direction', 'wave_sin_direction', 'wave_period_s', 'wave_energy', 'wave_power_kW_m'
]

N_CLUSTERS = 6
EXTREME_THRESHOLD = 0.9

In [0]:
scaler_path = f'/Volumes/cor_{ambiente}/ml/models/scaler/scaler_{{}}.pkl'

In [0]:
coast_names = (
    spark.sql(
        f"""
            SELECT DISTINCT coast_name
            FROM cor_{ambiente}.silver.swell_metrics
        """
    )
).toPandas()['coast_name'].tolist()

In [0]:
def prueba(features, sort_features, extreme_features):
    for coast in coast_names:
        print(f'Procesando costa: {coast}')
        # obtener datos
        data = (
            spark.sql(
                f"""
                    SELECT coast_name, datetime, {', '.join(SCALER_FEATURES)},
                    CONCAT(coast_name, '_', DATE_FORMAT(datetime, 'yyyyMM')) AS coast_year_month
                    FROM cor_{ambiente}.silver.swell_metrics
                    WHERE coast_name = '{coast}'
                    AND YEAR(datetime) % {EVERY_N_YEARS} = 0
                """
            )
        )

        if data.count() == 0:
            print(f'No hay datos para la costa {coast}, saltando...')
            continue

        # Generar data sample
        coast_year_month_dict = {row.coast_year_month: PORCENTAJE_SAMPLE_DATA for row in data.select('coast_year_month').distinct().collect()}
        data_sample = (
            data
            .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=RANDOM_SEED)
            .drop('coast_year_month')
        ).toPandas()

        # Preparar datos
        scaler = joblib.load(scaler_path.format(coast))
        scaled_data = scaler.transform(data_sample[SCALER_FEATURES])
        scaled_df = pd.DataFrame(scaled_data, columns=SCALER_FEATURES)

        # Separar datos en entrenamiento y prueba
        X = scaled_df[features]
        X_train, X_test = train_test_split(X, test_size=TEST_SIZE, random_state=RANDOM_SEED)

        # Entrenar modelo GMM de clasificación no supervisada
        gmm = GaussianMixture(
            n_components=N_CLUSTERS,
            covariance_type="full",
            random_state=RANDOM_SEED
        )
        gmm.fit(X_train)
        X_test["gmm_cluster"] = gmm.predict(X_test)
        X_test["gmm_cluster_probability"] = gmm.predict_proba(X_test).max(axis=1)

        # Evaluar modelo GMM
        q_5 = X_test.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.05)
        q_10 = X_test.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.10)
        q_15 = X_test.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.15)
        print(f'Quantiles: 0.05: {q_5}, 0.10: {q_10}, 0.15: {q_15}')
        if not (
            all(X_test.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.05)>=0.5) and
            all(X_test.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.10)>=0.7) and
            all(X_test.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.15)>=0.8) 
        ):
            print(f'El modelo GMM para la costa {coast} no es lo suficientemente bueno, saltando...')
            return
        else:
            print(f'El modelo GMM para la costa {coast} es bueno')
    
        

In [0]:
GMM_FEATURES = [
    'wind_speed_ms', 
    'wave_energy', 'wave_height_m', 'wave_period_s', 'wave_power_kW_m'
]
EXTREME_FEATURES = ['wind_speed_ms', 'wave_energy', 'wave_period_s']
SORT_FEATURES = ["wave_energy", "wave_period_s", "wind_speed_ms"]

prueba()

In [0]:
GMM_FEATURES = [
    'wind_speed_ms', 'wind_cos_direction', 'wind_sin_direction', 
    'wave_height_m', 'wave_cos_direction', 'wave_sin_direction', 'wave_period_s', 'wave_energy', 'wave_power_kW_m'
]
EXTREME_FEATURES = ['wind_speed_ms', 'wave_power_kW_m', 'wave_period_s']
SORT_FEATURES = ['wind_speed_ms', 'wave_power_kW_m', 'wave_period_s']

prueba()

In [0]:
GMM_FEATURES = [
    'wind_speed_ms', 
    'wave_energy', 'wave_power_kW_m'
]
EXTREME_FEATURES = ['wind_speed_ms', 'wave_power_kW_m', 'wave_energy']
SORT_FEATURES = ['wind_speed_ms', 'wave_power_kW_m', 'wave_energy']

prueba()

In [0]:
GMM_FEATURES = [
    'wind_speed_ms', 
    'wave_height_m', 'wave_period_s'
]
EXTREME_FEATURES = ['wind_speed_ms', 'wave_height_m', 'wave_period_s']
SORT_FEATURES = ['wind_speed_ms', 'wave_height_m', 'wave_period_s']

prueba()

In [0]:
GMM_FEATURES = [
    'wind_speed_ms', 
    'wave_height_m', 'wave_period_s', 'wave_energy', 'wave_power_kW_m'
]
EXTREME_FEATURES = ['wind_speed_ms', 'wave_power_kW_m', 'wave_period_s']
SORT_FEATURES = ['wind_speed_ms', 'wave_power_kW_m', 'wave_period_s']

prueba()

In [0]:
GMM_FEATURES = [
    'wind_speed_ms', 'wind_cos_direction', 'wind_sin_direction', 
    'wave_cos_direction', 'wave_sin_direction', 'wave_energy', 'wave_power_kW_m'
]
EXTREME_FEATURES = ['wind_speed_ms', 'wave_power_kW_m', 'wave_energy']
SORT_FEATURES = ['wind_speed_ms', 'wave_power_kW_m', 'wave_energy']

prueba()

In [0]:
GMM_FEATURES = [
    'wave_energy', 'wave_power_kW_m'
]
EXTREME_FEATURES = ['wave_energy', 'wave_power_kW_m']
SORT_FEATURES = ['wave_energy', 'wave_power_kW_m']

prueba()

In [0]:
GMM_FEATURES = [
    'wind_speed_ms', 'wind_cos_direction', 'wind_sin_direction', 
    'wave_height_m', 'wave_cos_direction', 'wave_sin_direction', 'wave_period_s'
]
EXTREME_FEATURES = ['wind_speed_ms', 'wave_height_m', 'wave_period_s']
SORT_FEATURES = ['wind_speed_ms', 'wave_height_m', 'wave_period_s']

prueba()

In [0]:
GMM_FEATURES = [
    'wind_speed_ms', 'wind_cos_direction', 'wind_sin_direction', 
    'wave_height_m', 'wave_cos_direction', 'wave_sin_direction', 'wave_period_s'
]
EXTREME_FEATURES = ['wind_speed_ms', 'wave_height_m', 'wave_period_s']
SORT_FEATURES = ['wind_speed_ms', 'wave_height_m', 'wave_period_s']

def prueba_rf():
    for coast in coast_names:
        print(f'Procesando costa: {coast}')
        # obtener datos
        data = (
            spark.sql(
                f"""
                    SELECT coast_name, datetime, {', '.join(SCALER_FEATURES)},
                    CONCAT(coast_name, '_', DATE_FORMAT(datetime, 'yyyyMM')) AS coast_year_month
                    FROM cor_{ambiente}.silver.swell_metrics
                    WHERE coast_name = '{coast}'
                    AND YEAR(datetime) % {EVERY_N_YEARS} = 0
                """
            )
        )

        if data.count() == 0:
            print(f'No hay datos para la costa {coast}, saltando...')
            continue

        # Generar data sample
        coast_year_month_dict = {row.coast_year_month: PORCENTAJE_SAMPLE_DATA for row in data.select('coast_year_month').distinct().collect()}
        data_sample = (
            data
            .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=RANDOM_SEED)
            .drop('coast_year_month')
        ).toPandas()

        scaler = joblib.load(scaler_path.format(coast))
        scaled_data = scaler.transform(data_sample[SCALER_FEATURES])
        scaled_df = pd.DataFrame(scaled_data, columns=SCALER_FEATURES)

        # Entrenar modelo GMM de clasificación no supervisada
        gmm = GaussianMixture(
            n_components=N_CLUSTERS,
            covariance_type="full",
            random_state=RANDOM_SEED
        )
        data_sample["gmm_cluster"] = gmm.fit_predict(scaled_df[GMM_FEATURES])
        data_sample["gmm_cluster_probability"] = gmm.predict_proba(scaled_df[GMM_FEATURES]).max(axis=1)
        
        cluster_summary = (
            data_sample.groupby("gmm_cluster")[GMM_FEATURES]
            .mean()
            .sort_values(SORT_FEATURES)
        )
        cluster_order = {
            old_cluster: new_cluster + 1
            for new_cluster, old_cluster in enumerate(cluster_summary.index)
        }

        data_sample["gmm_sea_state_level"] = data_sample["gmm_cluster"].map(cluster_order)

        data_sample['gmm_mask_extremo'] = data_sample[EXTREME_FEATURES[0]] > data_sample[EXTREME_FEATURES[0]].quantile(EXTREME_THRESHOLD)
        for feature in EXTREME_FEATURES[1:]:
            data_sample['gmm_mask_extremo'] &= data_sample[feature] > data_sample[feature].quantile(EXTREME_THRESHOLD)

        data_sample.loc[data_sample['gmm_mask_extremo'], 'gmm_sea_state_level'] = 7

        # Evaluar modelo GMM
        q_5 = data_sample.groupby('gmm_sea_state_level')['gmm_cluster_probability'].quantile(0.05)
        q_10 = data_sample.groupby('gmm_sea_state_level')['gmm_cluster_probability'].quantile(0.10)
        q_15 = data_sample.groupby('gmm_sea_state_level')['gmm_cluster_probability'].quantile(0.15)
        if not (
            all(data_sample.groupby('gmm_sea_state_level')['gmm_cluster_probability'].quantile(0.05)>=0.6) and
            all(data_sample.groupby('gmm_sea_state_level')['gmm_cluster_probability'].quantile(0.10)>=0.7) and
            all(data_sample.groupby('gmm_sea_state_level')['gmm_cluster_probability'].quantile(0.15)>=0.8) 
        ):
            print(f'El modelo GMM para la costa {coast} no es lo suficientemente bueno, saltando...')
            continue
        else:
            print(f'Modelo GMM para la costa {coast} cumple con los criterios')

        # Entrar modelo Random Forest de clasificación supervisada
        X = data_sample[RF_FEATURES]
        y = data_sample[TARGET].astype(int)
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            random_state=RANDOM_SEED,
            stratify=y
        )

        rf_classifier = RandomForestClassifier(
            n_estimators=N_ESTIMATORS,
            max_depth=MAX_DEPTH,
            min_samples_leaf=MIN_SAMPLES_LEAF,
            class_weight=CLASS_WEIGHT,
            random_state=RANDOM_SEED,
            n_jobs=N_JOBS
        )

        rf_classifier.fit(X_train, y_train)

        y_pred = rf_classifier.predict(X_test)

        # Evaluar modelo
        acurracy = accuracy_score(y_test, y_pred)
        balanced_acurracy = balanced_accuracy_score(y_test, y_pred)
        print(f'Accuracy: {acurracy}, Balanced Accuracy: {balanced_acurracy}')
        if not(
            acurracy >= 0.7 and 
            balanced_acurracy >= 0.5
        ):
            print(f'El modelo Random Forest para la costa {coast} no es lo suficientemente bueno, saltando...')
            return
        else:
            print(f'Modelo para la costa {coast} cumple con los criterios')

        y_train_pred = rf_classifier.predict(X_train)
        y_test_pred = rf_classifier.predict(X_test)

        print("Train accuracy:", accuracy_score(y_train, y_train_pred))
        print("Test accuracy:", accuracy_score(y_test, y_test_pred))

        print("Train balanced accuracy:", balanced_accuracy_score(y_train, y_train_pred))
        print("Test balanced accuracy:", balanced_accuracy_score(y_test, y_test_pred))


In [0]:
RF_FEATURES = [
    'wind_speed_ms', 'wind_cos_direction', 'wind_sin_direction', 
    'wave_height_m', 'wave_cos_direction', 'wave_sin_direction', 'wave_period_s'
]
TARGET = "gmm_sea_state_level"
TEST_SIZE = 0.25
N_ESTIMATORS = 500
MAX_DEPTH = None
MIN_SAMPLES_LEAF = 10
CLASS_WEIGHT = "balanced"
N_JOBS = -1

prueba_rf()